# ArtifactBench 1.2-rc2: reproduce statistics from public predictions

This notebook requires completed, hash-bound predictions for all eight models. It does not run audio inference or download recordings or weights. It recomputes the original statistics, including all bootstrap intervals and paired model differences, and requires byte-identical statistical JSON. Missing results are an error, never replaced with examples.

Historical smoke inputs are accepted only with an explicit test flag, produce conspicuous warnings, and are not final benchmark evidence. Official demonstrations remain separate from population-level metrics.

In [ ]:
import json
import os
from pathlib import Path
import sys
import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'artifactbench/v12/public_results.py').is_file())
sys.path.insert(0, str(ROOT))
from artifactbench.v12.public_results import load_public_results, reproduce_results
from artifactbench.v12.report import MODELS
assert Path(sys.modules[load_public_results.__module__].__file__).resolve().is_relative_to(ROOT.resolve()), 'Import outside source root'
if os.environ.get('ARTIFACTBENCH_NOTEBOOK_PYTHON'):
    assert Path(sys.executable).absolute() == Path(os.environ['ARTIFACTBENCH_NOTEBOOK_PYTHON']).absolute(), 'Wrong kernel executable'
if os.environ.get('ARTIFACTBENCH_NOTEBOOK_PREFIX'):
    assert Path(sys.prefix).resolve() == Path(os.environ['ARTIFACTBENCH_NOTEBOOK_PREFIX']).resolve(), 'Wrong kernel environment'
RELEASE = Path(os.environ.get('ARTIFACTBENCH_RELEASE', str(ROOT / 'out/v1.2_frozen_rc2_260905')))
PREDICTIONS = Path(os.environ.get('ARTIFACTBENCH_PUBLIC_PREDICTIONS', str(ROOT / 'out/v1.2_public_results_rc2_260905')))
OUTPUT = Path(os.environ.get('ARTIFACTBENCH_RESULTS_OUTPUT', str(ROOT / 'out/reproduced-public-statistics')))
SMOKE = os.environ.get('ARTIFACTBENCH_ALLOW_SMOKE') == '1'
BANNER = 'SMOKE WIRING ONLY: NOT BENCHMARK RESULTS' if SMOKE else 'Completed rc2 saved-score reproduction'
print(BANNER)
print('Python', sys.version.split()[0], 'NumPy', np.__version__)

## Verify input identity and coverage

Every exported file is checked against its hash inventory. Prediction IDs, labels, audio hashes, source cells, recording representatives, and dependence clusters must match the public release. Failures remain explicit and cannot carry invented probabilities.

In [ ]:
public, runs, identities, export_summary = load_public_results(PREDICTIONS, RELEASE, SMOKE)
if not SMOKE:
    assert export_summary['manifest_sha256'] == 'feab7c4c3d037919fd784dc40e46199470088abde532c5181e34573f20dceefd', 'Not the pinned rc2 release'
print(BANNER)
for name in MODELS:
    rows = list(runs[name].values())
    scored = sum(r['outcome'] == 'scored' for r in rows)
    print(f'{name:20s} scored={scored} attempted={len(rows)} failures={len(rows)-scored}')
print('Real controls are all legacy; independent contemporary real-audio FPR is undefined.')

## Recompute every statistical file

The full mode uses 2,000 deterministic recording/creator/lineage cluster bootstrap replicates. Source/label cells linked by one dependence cluster are resampled together. Historical smoke mode uses only 20 replicates to test the code path. A numerical mismatch with any original file is an error. The percentile convention is NumPy's [linear quantile method](https://numpy.org/doc/2.1/reference/generated/numpy.quantile.html).

In [ ]:
print(BANNER)
verification = reproduce_results(PREDICTIONS, RELEASE, OUTPUT, SMOKE)
assert verification['compared_statistical_files'] == 9
print(json.dumps(verification, indent=2))
reports = {name: json.loads((OUTPUT / (name + '.json')).read_text()) for name in MODELS}
def fmt(metric):
    if metric['estimate'] is None:
        return 'undefined'
    return f"{metric['estimate']:.4f} [{metric['lower']:.4f}, {metric['upper']:.4f}]"

In [ ]:
print(BANNER)
for name in MODELS:
    report = reports[name]
    legacy = report['cohorts']['legacy']['metrics']
    suno = report['sources']['suno_v5.5_native_260905']['metrics']
    udio = report['sources']['udio_2026_version_unknown_native_260905']['metrics']
    print(name)
    print('  legacy F1:', fmt(legacy['F1']), 'legacy FPR:', fmt(legacy['FPR']))
    print('  native Suno TPR:', fmt(suno['TPR']), 'native Udio TPR:', fmt(udio['TPR']))
print('Intervals are descriptive within this sampled mixture, not universal deployment guarantees.')

In [ ]:
print(BANNER)
print('Existing ArtifactNet operating stacks, non-demo mixture; thresholds are not retuned:')
for name, result in reports['artifactnet']['operating_stacks'].items():
    print(name, {metric: fmt(result['metrics'][metric]) for metric in ('F1', 'TPR', 'FPR')})
paired = json.loads((OUTPUT / 'paired_differences.json').read_text())
assert len(paired) == 28
print('Verified paired model comparisons:', len(paired))
print('Differences use common-success recordings, not mismatched model denominators.')

In [ ]:
print(BANNER)
print('Provider-selected demonstrations: TP / scored / attempted, descriptive only')
for name in MODELS:
    for source, result in reports[name]['official_demonstrations'].items():
        point = result['point']
        print(name, source, f"{point['TP']:.0f}/{point['scored']:.0f}/{point['attempted']:.0f}")
print('Done: saved-score statistics only. Audio inference, transport evaluation, and rights remain separate checks.')

## What passing means

All eight saved prediction sets reproduce the original statistical JSON. It does not independently validate the source recordings, reproduce model execution, or establish training independence. Udio generator versions remain unknown; platform-generated-or-edited is not a universally verified fully synthetic label. The native pool was retrospectively observed, and official demos are provider-selected.

LaTeX paper tables and figures are generated separately from these completed statistics. Notebook execution follows the [official NBClient API](https://nbclient.readthedocs.io/en/latest/client.html); errors stop execution and actual outputs are preserved.